In [17]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import torch

In [8]:
data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/mme/mme_base_des_05_11_2025.csv")

In [9]:
data

,question,gt_answer,question_id,image_id,image_path,data_type,answer
0,Is there a blue court in the image? Please ans...,Yes,231eab02-8ebe-4b46-b47e-4ae24c83b59e,12120,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"Yes, there is a blue court in the image."
1,Is there a purple court in the image? Please a...,No,c7840ca0-1b16-4c44-8b86-a7c95b5ba0aa,12120,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"Yes, there is a purple court in the image."
2,Is there a red couch in the image? Please answ...,Yes,4d5a9202-def3-4fdc-a5a4-cdd2578f98a1,564280,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"Yes, there is a red couch in the image."
3,Is there a black couch in the image? Please an...,No,5d454889-4297-4834-8c81-9fa1f04dc95b,564280,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"No, there is no black couch in the image. The ..."
4,Is there a white plate in the image? Please an...,Yes,5969601a-b454-4085-b0a0-e760126a0414,8277,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"Yes, there is a white plate in the image."
...,...,...,...,...,...,...,...
235,Is the mirror under the TV? Please answer yes ...,No,58fd3418-20ef-47c4-a184-8fc05595eb73,509699,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,position,Yes
236,Is the blue umbrella under the black umbrella?...,Yes,c19fef74-366e-4ad0-913a-d9faf4d30f67,212800,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,position,Yes
237,Is the blue umbrella above the black umbrella?...,No,525866a7-3808-49ea-8625-2d786694f0f5,212800,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,position,Yes
238,Is there a sofa in the middle of potted plants...,Yes,1fb86a62-c990-4294-9068-c74a9b6ffe6a,31248,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,position,"No, there is no sofa in the middle of potted p..."


In [12]:
preds = []
for i in data["answer"]:
    pred = i[:10].lower()
    if "yes" in pred:
        preds.append("Yes")
    elif "no" in pred:
        preds.append("No")
    else:
        preds.append("unknown")

In [13]:
pd.Series(preds).value_counts()

Yes    128
No     112
Name: count, dtype: int64

In [14]:
accuracy_score(data["gt_answer"], preds)

0.8666666666666667

In [66]:
detection_result_df = pd.read_pickle("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/mme/mme_llava_label_with_evidence_and_attn_detection_05_11_2025.pkl")

In [ ]:
new_preds = []
hal = 0
for inx, row in detection_result_df.iterrows():
    res = row["labels_with_evidence"]
    old_pred = row["answer"][:10].lower()
    
    hal_words = []
    for i in res[:1]:
        viz_evi = (torch.tensor(i["evidence"]) >= 0.5).int().sum().item()
        prob = i["label"]
        if prob <= 0.65:
            hal_words.append(i["word"])
        elif viz_evi <= 5 and prob <= 0.75:
            hal_words.append(i["word"])
        
        # if viz_evi <= 5 and prob <= 0.65:
        #     hal_words.append(i["word"])
    
    if not hal_words:
        # print(hal_words)
        if "yes" in old_pred:
            pred = "Yes"
        elif "no" in old_pred:
            pred = "No"
    else:
        hal += 1
        if "yes" in old_pred:
            pred = "No"
        elif "no" in old_pred:
            pred = "Yes"

    new_preds.append(pred)


In [72]:
accuracy_score(data["gt_answer"], new_preds)

0.8666666666666667